# Developing python implementation of BASL

## Understanding `biasAwareSelfLearning`

The signature:

```r
biasAwareSelfLearning <- function(accepts,
                                  rejects,
                                  target           = 'BAD', 
                                  filtering_beta   = c(0, 1),
                                  weak_learner     = 'classif.logreg',
                                  strong_learner   = 'classif.logreg',
                                  holdout_percent  = 0.1,
                                  sampling_percent = 1,
                                  labeling_percent = 0.01, 
                                  multiplier       = 1, 
                                  max_iterations   = 3,
                                  early_stop       = F,
                                  silent           = F) 
```

The call
```r
# BASL parameters
  filtering_beta   <- c(0.01, 0.99)
  holdout_percent  <- 0.1
  labeling_percent <- 0.1
  sampling_percent <- 0.8
  multiplier       <- 2
  max_iterations   <- 5
  early_stop       <- T
  
  # label rejected cases
  rej_labels <- biasAwareSelfLearning(accepts          = current_accepts, 
                                      rejects          = current_rejects,
                                      target           = 'BAD', 
                                      filtering_beta   = filtering_beta,
                                      weak_learner     = 'classif.logreg',
                                      strong_learner   = 'classif.logreg',
                                      holdout_percent  = holdout_percent,
                                      labeling_percent = labeling_percent, 
                                      sampling_percent = sampling_percent,
                                      multiplier       = multiplier, 
                                      max_iterations   = max_iterations,
                                      early_stop       = early_stop,
                                      silent           = T)
```

There are a couple of parts in the algorithm:

0. **Preparation:** defines several variables. 
    * specially interesting is that it creates "holdout samples" from the accepts when `early_stop == T`
    * it also defines as `bm_metric` the auc.
1. **Reject Filtering** It occurs if `filtering_beta != c(0,1)` (as the call is done inside the acceptance loop). The filtering is done via the function `filteringStage`

In [ ]:
from sklearn.ensemble import IsolationForest
from typing import Callable
class BaseBASL:
    def __init__(
            self,
            filtering_beta : torch.Tensor,
            weak_learner : Callable[[torch.Tensor], torch.Tensor],   # Consider making abstract class Learners
            strong_learner : Callable[[torch.Tensor], torch.Tensor], # Consider making abstract class Learners,
            hold_out_percent : float,
            labeling_percent : float,
            multiplier : float,
            max_iterations :int,
            early_stop : bool,
            isolation_forest : IsolationForest
    ):
        pass

    def self_learn(
            self,
            features_accept : torch.Tensor,
            default_flags_accept : torch.Tensor,
            features_reject : torch.Tensor,
            silent : bool = True
    ) -> torch.Tensor:
        pass

    def filter_rejects(
            self,
            features_accept : torch.Tensor,
            features_reject : torch.Tensor,
            *args,
            **kwargs
    ) -> torch.Tensor:
        pass

    def bayesian_evaluation( #Next step to implement
            self,
            features_accept : torch.Tensor,
            default_flag_accept : torch.Tensor,
            features_reject : torch.Tensor,
            *args,
            **kwargs
    ) -> torch.Tensor:
        pass
    
    def should_stop_early(self) -> bool:
        pass
    
    def label_rejects(self) -> torch.Tensor:
        pass
    
    
    


### Understanding `filteringStage`

Signature:

```r
filteringStage <- function(accepts, 
                           rejects, 
                           target    = 'BAD', 
                           beta      = c(0, 1),
                           num_trees = 100)
```

Call:

```r
filter_idx <- filteringStage(accepts   = train, # ifelse(early_stop, accepts[-holdout_idx_accepts, ], accepts)
                             rejects   = test,  # ifelse(early_stop, rejects[-holdout_idx_rejects, ], rejects)
                             beta      = filtering_beta, # From loop: c(0.01,0.99)
                             num_trees = 100)
```

In [ ]:
from sklearn.ensemble import IsolationForest
import numpy as np

forest = IsolationForest(n_estimators=100, max_samples="auto", random_state=1807)
def filter(
    ifo: IsolationForest,
    lower_trim_quantile: float,
    upper_trim_quantile: float,
    features: np.ndarray,
    return_index: bool = True,
) -> Union[np.ndarray, np.ndarray]:
    """
    Filters observations using a two-sided trimming strategy based on
    Isolation Forest normality scores.

    The function fits an Isolation Forest model on the provided feature
    matrix and computes the negative anomaly scores via
    ``IsolationForest.score_samples``. These scores induce a relative
    normality (similarity) ranking.

    Observations are retained if their score lies within the central
    quantile interval defined by ``lower_trim_quantile`` and
    ``upper_trim_quantile``. Consequently, both highly anomalous
    observations (lower tail) and overly typical observations
    (upper tail) are removed.

    This procedure corresponds to a two-sided percentile-based filtering
    scheme as described in Kozodoi et al. (2025), "Fighting Sampling Bias".

    Args:
        ifo (IsolationForest):
            An unfit ``IsolationForest`` instance used to compute
            normality scores.
        lower_trim_quantile (float):
            Lower quantile boundary in the interval ``[0, 1]``.
            Observations with scores below this quantile are discarded.
        upper_trim_quantile (float):
            Upper quantile boundary in the interval ``[0, 1]``.
            Observations with scores above this quantile are discarded.
        features (np.ndarray):
            Feature matrix of shape ``(n_samples, n_features)``.
        return_index (bool, optional):
            If ``True``, return a boolean mask indicating retained
            observations. If ``False``, return the filtered feature
            matrix. Defaults to ``True``.

    Returns:
        np.ndarray:
            If ``return_index`` is ``True``, a boolean array of shape
            ``(n_samples,)`` indicating which observations are retained.
            Otherwise, a feature matrix containing only the retained
            observations.

    Raises:
        ValueError:
            If ``lower_trim_quantile`` or ``upper_trim_quantile`` are
            outside the interval ``[0, 1]`` or if
            ``lower_trim_quantile >= upper_trim_quantile``.
    """
    if not 0.0 <= lower_trim_quantile <= 1.0:
        raise ValueError("lower_trim_quantile must be in the interval [0, 1].")
    if not 0.0 <= upper_trim_quantile <= 1.0:
        raise ValueError("upper_trim_quantile must be in the interval [0, 1].")
    if lower_trim_quantile >= upper_trim_quantile:
        raise ValueError(
            "lower_trim_quantile must be strictly smaller than upper_trim_quantile."
        )

    ifo.fit(features)
    normality_scores = ifo.score_samples(features)

    lower_score_bound, upper_score_bound = np.quantile(
        normality_scores,
        [lower_trim_quantile, upper_trim_quantile],
    )

    keep_mask = (
        (lower_score_bound <= normality_scores)
        & (normality_scores <= upper_score_bound)
    )

    return keep_mask if return_index else features[keep_mask]
keep_mask = filter(forest, beta_bottom=0.01, beta_top=0.99, features=features.numpy())
